In [1]:
from tensorflow.keras.datasets import reuters
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer


# 모델
from sklearn.linear_model import LogisticRegression # softmax
from sklearn.svm import SVC, LinearSVC  # Linear svc, SVN RBF kernel 2개
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import ComplementNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import VotingClassifier

# 지표
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score

In [2]:
# 데이터 로드
(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=10000, test_split=0.2)

word_index = reuters.get_word_index()
index_to_word = {i+3: word for word, i in word_index.items()}
for i, word in enumerate(("<pad>", "<sos>", "<unk>")):
    index_to_word[i] = word

# 데이터 복원
def decode_news(sequence):
    return ' '.join([index_to_word.get(i, '<unk>') for i in sequence])    # 리스트 컴프리헨션으로

x_train_text = [decode_news(seq) for seq in x_train]
x_test_text = [decode_news(seq) for seq in x_test]

# 벡터화
tfidf = TfidfVectorizer() # DTM + TF-IDF 동시에 구현
tfidfv_train = tfidf.fit_transform(x_train_text)
tfidfv_test = tfidf.transform(x_test_text)

| Vocabulary Size | Model | Accuracy | F1-Score | Best F1 |
| :--- | :--- | :---: | :---: | :---: |
| 10,000 | LogisticRegression | - | - | - |
| 10,000 | SVM (RBF Kernel) | - | - | - |
| 10,000 | Linear SVC |-  |-  | - |
| 10,000 | RandomForest | - | - | - |
| 10,000 | ComplementNB | - | - | - |
| 10,000 | XGBoost | - | - | - |
| 10,000 | LightGBM | - |- | - |
| 10,000 | Dense (Deep Learning) |  -| - | - |
| 10,000 | Voting (Soft) | - | - |-  |

In [3]:
# 모델 학습

model = {
    'LogisticRegression': LogisticRegression(penalty='l2', solver='liblinear', random_state=42),
    'SVC': SVC(kernel='linear', probability=True, random_state=42),
    'LinearSVC': LinearSVC(random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'ComplementNB': ComplementNB(),
    'XGBoost': XGBClassifier(random_state=42, n_jobs = -1),
    # 'LightGBM': LGBMClassifier(random_state=42) -> 성능이 너무 별로라 측정 후 제외
    }

for name, model in model.items():
    print(f"{name} 훈련 시작~")
    model.fit(tfidfv_train, y_train)
    y_pred = model.predict(tfidfv_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"{name} Accuracy: {accuracy:.4f}")
    print(f"{name} F1-Score (Weighted): {f1:.4f}")
    print("-------------------------------------")

LogisticRegression 훈련 시작~
LogisticRegression Accuracy: 0.7876
LogisticRegression F1-Score (Weighted): 0.7613
-------------------------------------
SVC 훈련 시작~
SVC Accuracy: 0.8219
SVC F1-Score (Weighted): 0.8121
-------------------------------------
LinearSVC 훈련 시작~
LinearSVC Accuracy: 0.8299
LinearSVC F1-Score (Weighted): 0.8237
-------------------------------------
RandomForest 훈련 시작~
RandomForest Accuracy: 0.7542
RandomForest F1-Score (Weighted): 0.7295
-------------------------------------
ComplementNB 훈련 시작~
ComplementNB Accuracy: 0.7707
ComplementNB F1-Score (Weighted): 0.7457
-------------------------------------
XGBoost 훈련 시작~
XGBoost Accuracy: 0.7961
XGBoost F1-Score (Weighted): 0.7895
-------------------------------------


In [4]:
# Voting의 경우
from sklearn.ensemble import VotingClassifier
from sklearn.calibration import CalibratedClassifierCV

# 성능 상위 모델 조합
voting_model = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(C=10, solver='liblinear', random_state=42)),
        ('cb', ComplementNB()),
        ('SVC', SVC(kernel='linear', probability=True, random_state=42)),
        ('xgb', XGBClassifier(n_estimators=100, random_state=42))
    ],
    voting='soft',
    n_jobs=-1
)


# 학습 및 평가
print("voting 모델 학습 및 평가 중~")
voting_model.fit(tfidfv_train, y_train)

y_pred_vt = voting_model.predict(tfidfv_test)
accuracy_vt = accuracy_score(y_test, y_pred)
f1_vt = f1_score(y_test, y_pred, average='weighted')
print(f"Voting 모델 정확도: {accuracy_vt}")
print(f"Voting 모델 F1 Score: {f1_vt}")

voting 모델 학습 및 평가 중~
Voting 모델 정확도: 0.7960819234194123
Voting 모델 F1 Score: 0.7894532460248317


In [6]:
# Dense 레이어 (DL) 구현해보기
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, f1_score
from tensorflow.keras import regularizers
from sklearn.utils import class_weight
import numpy as np

# 1. 실제 데이터 크기 확인 (9670 등 실제 feature 수)
input_dim = tfidfv_train.shape[1]


model = Sequential([
    Dense(512, activation='relu', input_dim=input_dim),
    # kernel_regularizer를 추가하여 가중치를 억제합니다.
    Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    BatchNormalization(),
    Dropout(0.5),

    Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    BatchNormalization(),
    Dropout(0.5),

    Dense(46, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 클래스별 가중치 계산
weights = class_weight.compute_class_weight('balanced',
                                            classes=np.unique(y_train),
                                            y=y_train)
class_weights = dict(enumerate(weights))

es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# 학습 시 적용
history = model.fit(tfidfv_train.toarray(), y_train,
                    epochs=20,
                    batch_size=128,
                    validation_split=0.2,
                    class_weight=class_weights, # 가중치 적용!
                    callbacks=[es])


# 3. 학습 (Scipy sparse matrix 에러 방지를 위해 .toarray() 사용)
history = model.fit(tfidfv_train.toarray(), y_train,
                    epochs=20, # EarlyStopping이 있으므로 에폭을 넉넉히 둡니다.
                    batch_size=128,
                    validation_split=0.2, # 검증 데이터 비율 상향
                    callbacks=[es],
                    verbose=1)

# 4. 예측 및 평가
y_pred_probs = model.predict(tfidfv_test.toarray())
y_pred = np.argmax(y_pred_probs, axis=1)

print(f"\n[Deep Learning Optimized Results]")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-Score (Weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 8s 92ms/step - accuracy: 0.1023 - loss: 5.2177 - val_accuracy: 0.0089 - val_loss: 4.7033
Epoch 2/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 5s 90ms/step - accuracy: 0.4421 - loss: 2.4428 - val_accuracy: 0.1564 - val_loss: 4.5677
Epoch 3/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 5s 89ms/step - accuracy: 0.6181 - loss: 1.5693 - val_accuracy: 0.3211 - val_loss: 4.1259
Epoch 4/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 5s 84ms/step - accuracy: 0.7347 - loss: 1.2634 - val_accuracy: 0.4524 - val_loss: 3.5969
Epoch 5/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 6s 94ms/step - accuracy: 0.7931 - loss: 1.1320 - val_accuracy: 0.5832 - val_loss: 3.0217
Epoch 6/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 5s 86ms/step - accuracy: 0.8618 - loss: 0.9965 - val_accuracy: 0.6500 - val_loss: 2.5401
Epoch 7/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 5s 89ms/step - accuracy: 0.8858 - loss: 0.9528 - val_accuracy: 0.7067 - val_loss: 2.0900
Epoch 8/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 5s 79ms/step - accuracy: 0.8967 - loss: 0.8831 - val_accuracy: 0.7457 - v